### Import Dependencies

In [3]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
import os
from dotenv import load_dotenv

from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents.run_config import RunConfig

from google.genai import types

load_dotenv("../../.env")

from utils.tools_2 import check_warehouse_availability, reserve_warehouse_items

### ADK Agent

# built to work googles gemini models so we need to use some abstraction frameworks that allows us to abstrafct google api on top of other providers

In [ ]:
model = LiteLlm(
    model="openai/gpt-5.4-mini",
    api_key=os.environ.get("OPENAI_API_KEY")
)
warehouse_agent = Agent(
    name="warehouse_manager_agent",
    description="The user is asking to reserve items from the warehouses or about availability of items in the warehouses.",
    instruction="""You are a part of the shopping assistant that can manage available inventory in the warehouses.

    ## Instructions

    - As the final answer you should return an answer to the users query in a form of actions performed.
    - You must always check the availability of the items in the warehouses before reserving them.
    - Only reserve items in warehouses if entire order can be reserved or the user has confirmed that they want a partial reservation.
    - If you cannot reserve any items, return an answer that the order cannot be reserved.
    - If you can reserve some items, return an answer that the order can be partially reserved and include the details.
    - If only partial quantity can be reserved in some warehouses, try to combine the required quantity from different war
    - Try to reserve items from the closest warehouse to the user first if users location is provided.
    - As the final answer you should return an answer in a form of actions performed.
    """,
    model=model,
    tools=[
        check_warehouse_availability,
        reserve_warehouse_items
    ]
)

In [6]:
session_service = InMemorySessionService()
await session_service.create_session(
    app_name="warehouse_app",
    user_id="user_1",
    session_id="session_1"
)
runner = Runner(
    agent=warehouse_agent,
    session_service=session_service,
    app_name="warehouse_app"
)
message = types.Content(
    role="user",
    parts=[
        types.Part(
            text="What is the availability of B09X1LDMH6 in all of your warehouses"
        )
    ]
)
result = runner.run(
    user_id="user_1",
    session_id="session_1",
    new_message=message,
    run_config=RunConfig(
        max_llm_calls=5
    )
)
for event in result:
    print("========")
    print(event)

/Users/tvelev/sc/swirl-bc/.venv/lib/python3.12/site-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


model_version='gpt-5.4-mini-2026-03-17' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'items': [
            {<... 2 items at Max depth ...>},
          ]
        },
        id='call_yF5zard4ZcCNZDli1ekmYz2D',
        name='check_warehouse_availability'
      )
    ),
  ],
  role='model'
) grounding_metadata=None partial=False turn_complete=None turn_complete_reason=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  cached_content_token_count=0,
  candidates_token_count=36,
  prompt_token_count=652,
  total_token_count=688
) live_session_resumption_update=None live_session_id=None go_away=None voice_activity=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None environment_id=None invocation_id='e-8dc226c7-abbd-4

### Agent Run Wrapper

In [8]:
async def run_warehouse_agent(query: str, session_id: str, session_service: InMemorySessionService) -> str:
    existing_session = await session_service.get_session(
        app_name="warehouse_app",
        user_id="user_1",
        session_id=session_id
    )
    
    if not existing_session:
        await session_service.create_session(
                app_name="warehouse_app",
                user_id="user_1",
                session_id=session_id
            )

    runner = Runner(
        agent=warehouse_agent,
        session_service=session_service,
        app_name="warehouse_app"
    )
    
    content = types.Content(
        role="user",
        parts=[
            types.Part(
                text=query
            )
        ]
    )
    
    final_answer = ""
    for event in runner.run(
        user_id="user_1",
        session_id=session_id,
        new_message=content,
        run_config=RunConfig(
            max_llm_calls=5
        )
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    final_answer += part.text
    
    return final_answer

In [9]:
answer = await run_warehouse_agent(
    query="What is the availability of B09X1LDMH6 in all of your warehouses",
    session_id="test_01", # if we provide the same session id we will continue the conversation
    session_service=session_service
)

In [11]:
print(answer)

Availability checked for **B09X1LDMH6**:

- **Hamburg North Warehouse (DE-HAM-01), Hamburg, Germany**: **22 units available**
- This warehouse can fulfill the item completely.

**Result:** The item is available in our warehouses and can be fulfilled in full.


In [12]:
answer_2 = await run_warehouse_agent(
    query="Can you reserve 3 of this B09X1LDMH6 in Hamburg",
    session_id="test_01", # if we provide the same session id we will continue the conversation
    session_service=session_service
)

In [13]:
print(answer_2)

Actions performed:

- Checked availability of **B09X1LDMH6** in Hamburg warehouse(s)
- Confirmed **3 units** are available in **Hamburg North Warehouse (DE-HAM-01)**
- Reserved **3 units** successfully

Reservation details:
- Product: **B09X1LDMH6**
- Quantity reserved: **3**
- Warehouse: **Hamburg North Warehouse**
- Warehouse ID: **DE-HAM-01**
